# Supervised Fine-Tuning with SFTTrainer

This notebook demonstrates how to fine-tune the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer` from the `trl` library. The notebook cells run and will finetune the model. You can select your difficulty by trying out different datasets.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Fine-Tuning SmolLM2 with SFTTrainer</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the `HuggingFaceTB/smoltalk` dataset</p>
    <p>🐕 Try out the `bigcode/the-stack-smol` dataset and finetune a code generation model on a specific subset `data/python`.</p>
    <p>🦁 Select a dataset that relates to a real world use case your interested in</p>
</div>

In [2]:
# Install the requirements in Google Colab
!pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face

from huggingface_hub import login
# login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [25]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Base models don't have a chat template. We'll set a basic one (ChatML format)
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"


# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset2"
finetune_tags = ["smol-course", "module_1"]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [26]:
device

'cuda'

In [24]:
del model,tokenizer

# Generate with the base model

Here we will try out the base model which does not have a chat template.

In [30]:
# Let's test the base model before training with the SAME sampling parameters
prompt = "I'm a little sad, can you give me some advice?"

# Format with template (base model doesn't know this, so it will just treat it as text)
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
# Using the exact same sampling settings as the 'after' test
# 显式获取停止令牌的 ID
stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]

outputs = model.generate(
    **inputs,
    max_new_tokens=200, # 减小上限，避免极端情况下的刷屏
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    eos_token_id=stop_token_ids, # 告诉模型这些是结束标志
    pad_token_id=tokenizer.pad_token_id
)

print("Before training (Base Model with Sampling):")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Before training (Base Model with Sampling):
user
I'm a little sad, can you give me some advice?
assistant
I can't believe you said that!
I don't know what to say, but I'm sorry.
Well, you have a good idea.
I'm sorry, I'm not going to do that.
I don't know what to say.
I'm sorry, I'm not going to do that.
What's the use of saying anything?
You should be able to go and have a drink with us.
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.
What are you doing?
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.
I'm sorry, I'm not going to do that.



## Dataset Preparation

We will load a sample dataset and format it for training. The dataset should be structured with input-output pairs, where each input is a prompt and the output is the expected response from the model.

**TRL will format input messages based on the model's chat templates.** They need to be represented as a list of dictionaries with the keys: `role` and `content`,.

In [7]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
ds = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/everyday-conversations/train-00000-(…): reconstructing file:   0%|          |  0.00B /  946kB            

data/everyday-conversations/train-00000-(…): downloading bytes:           |  0.00B            

data/everyday-conversations/test-00000-o(…): reconstructing file:   0%|          |  0.00B / 52.6kB            

data/everyday-conversations/test-00000-o(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

In [ ]:
# TODO: 🦁 If your dataset is not in a format that TRL can convert to the chat template, you will need to process it. Refer to the [module](../chat_templates.md)

## Configuring the SFTTrainer

The `SFTTrainer` is configured with various parameters that control the training process. These include the number of training steps, batch size, learning rate, and evaluation strategy. Adjust these parameters based on your specific requirements and computational resources.

In [18]:
from trl import SFTConfig,SFTTrainer

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=16,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    eval_strategy="steps",
    eval_steps=50,  # Frequency of evaluation
    hub_model_id=finetune_name,  # Set a unique name for your model
    completion_only_loss=True,
)

# Initialize the SFTTrainer
# Note: In latest TRL, 'processing_class' is often used, but 'tokenizer' should work
# if passed correctly as a positional or specific keyword depending on the version.
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer, # Using processing_class which is the new standard in TRL
)
# TODO: 🦁 🐕 align the SFTTrainer params with your chosen dataset.

Tokenizing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

## Training the Model

With the trainer configured, we can now proceed to train the model. The training process will involve iterating over the dataset, computing the loss, and updating the model's parameters to minimize this loss.

In [19]:
# Train the model
trainer.train()

# Save the model
trainer.save_model(f"./{finetune_name}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.569301,1.558004,1.633893,154355.000000,0.626964
100,1.297662,1.301471,1.476094,309790.000000,0.668314
150,1.171103,1.228160,1.388432,463299.000000,0.686857
200,1.187867,1.194625,1.334917,619369.000000,0.704406
250,1.135903,1.171849,1.302632,775668.000000,0.712266
300,1.130357,1.157066,1.282102,927951.000000,0.714197
350,1.083576,1.145369,1.268300,1082696.000000,0.714105
400,1.104839,1.137045,1.257867,1237987.000000,0.715803
450,1.095931,1.130899,1.240372,1391900.000000,0.716329
500,1.124730,1.128104,1.236373,1546875.000000,0.716522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.push_to_hub(tags=finetune_tags)

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Generate with fine-tuned model</h2>
    <p>🐕 Use the fine-tuned to model generate a response, just like with the base example..</p>
</div>

In [15]:
# Test the fine-tuned model on the same prompt

# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# TODO: use the fine-tuned to model generate a response, just like with the base example.

In [32]:
# Load the fine-tuned model for testing
from transformers import AutoModelForCausalLM

ft_model = AutoModelForCausalLM.from_pretrained(f"./{finetune_name}").to(device)

prompt = "I'm a little sad, can you give me some advice?"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# 显式获取停止令牌的 ID
stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]

outputs = ft_model.generate(
    **inputs,
    max_new_tokens=200, # 减小上限，避免极端情况下的刷屏
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    eos_token_id=stop_token_ids, # 告诉模型这些是结束标志
    pad_token_id=tokenizer.pad_token_id
)

print("After training (Fine-tuned Model with Stop Control):")
# 使用 skip_special_tokens=False 可以观察它是否输出了结束标签
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

After training (Fine-tuned Model with Stop Control):
user
I'm a little sad, can you give me some advice?
assistant
It's okay to be sad sometimes. You can always try to look for positive news or experiences. It's normal to feel disappointed when something doesn't go as planned.


In [77]:
ds["train"]

Dataset({
    features: ['full_topic', 'messages'],
    num_rows: 2260
})

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [36]:
!cp -r /content/SmolLM2-FT-MyDataset2 /content/drive/MyDrive/finetuend_models/

In [37]:
!ls /content/drive/MyDrive/finetuend_models/

chat_template.jinja	model.safetensors      training_args.bin
config.json		tokenizer_config.json
generation_config.json	tokenizer.json


## 💐 You're done!

This notebook provided a step-by-step guide to fine-tuning the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer`. By following these steps, you can adapt the model to perform specific tasks more effectively. If you want to carry on working on this course, here are steps you could try out:

- Try this notebook on a harder difficulty
- Review a colleagues PR
- Improve the course material via an Issue or PR.